In [3]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [4]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords , shakespeare

In [ ]:

nltk.download('shakespeare')
nltk.download('stopwords')

In [ ]:
shakespeare.fileids()

In [ ]:
stopwords.words('english')

In [ ]:
shakespeare.words(fileid='hamlet.xml')

In [8]:
# Inverted_index = {
#    word : [list of document ids where the word appears]  
# }
# document_ids = shakespeare.fileids() --> index
# 

In [9]:
type(shakespeare.fileids())

list

In [ ]:
# Bug 1 : The inverted_index[word] = ids is editing the value to the last seen doc_id 
# Bug 2 : The word.lower() does nothing as str are immutable
# No set used rn and no stopword removal rn

In [25]:
# inverted_index = {}
# file_ids = shakespeare.fileids()
# for ids , doc_ids in enumerate(file_ids):
#     print(doc_ids,ids)
#     for word in shakespeare.words(fileid=doc_ids) :
#         word.lower()
#         inverted_index[word] = ids
# print(inverted_index)

In [ ]:
# Bug 1 : The one line if else has problem dealing with mutating a data structure
# Bug 2 : Didnt handle stopwords

In [ ]:
# inverted_index = {}
# file_ids = shakespeare.fileids()
# for ids , doc_ids in enumerate(file_ids):
#     raw_doc = shakespeare.words(fileid=doc_ids)
#     for word in raw_doc :
#         word = word.lower()
#         if word not in inverted_index : inverted_index[word] = {ids}
#         else : inverted_index[word].add(ids)


{'the': {0, 1, 2, 3, 4, 5, 6, 7}, 'tragedy': {0, 1, 2, 3, 4, 6, 7}, 'of': {0, 1, 2, 3, 4, 5, 6, 7}, 'antony': {0, 3, 4, 7}, 'and': {0, 1, 2, 3, 4, 5, 6, 7}, 'cleopatra': {0, 7}, 'dramatis': {0, 1, 2, 3, 4, 5, 6, 7}, 'personae': {0, 1, 2, 3, 4, 5, 6, 7}, 'mark': {0, 1, 2, 3, 4, 5, 6, 7}, 'octavius': {0, 3}, 'caesar': {0, 2, 3, 4, 6}, 'm': {0, 2, 3, 6}, '.': {0, 1, 2, 3, 4, 5, 6, 7}, 'aemilius': {0, 3}, 'lepidus': {0, 3}, 'triumvirs': {0, 3}, 'sextus': {0}, 'pompeius': {0}, 'domitius': {0}, 'enobarbus': {0}, 'ventidius': {0}, 'eros': {0}, 'scarus': {0}, 'dercetas': {0}, 'demetrius': {0, 1}, 'philo': {0}, 'friends': {0, 1, 2, 3, 4, 5, 6, 7}, 'to': {0, 1, 2, 3, 4, 5, 6, 7}, 'mecaenas': {0}, 'agrippa': {0}, 'dolabella': {0}, 'proculeius': {0}, 'thyreus': {0}, 'gallus': {0}, 'menas': {0}, 'menecrates': {0}, 'varrius': {0}, 'pompey': {0, 3}, 'taurus': {0, 1}, ',': {0, 1, 2, 3, 4, 5, 6, 7}, 'lieutenant': {0, 6}, '-': {0, 1, 2, 3, 4, 5, 6, 7}, 'general': {0, 2, 3, 4, 5, 6, 7}, 'canidius': {0}, 

In [ ]:
# Bug 1 : used the res_set as empty set() so any intersection will give set() only
# Bug 2 : str.split() handles all whitespaces
# Handled the null set problem by seeding res_set() with first word doc_ids set

In [ ]:
# def basic_search1(inverted_index ,raw_query):
#     raw_query = raw_query.split() 
#     pro_query = [word.lower() for word in raw_query if word.lower() in inverted_index]
#     if not pro_query : return set()
#     result_set = inverted_index.get(pro_query[0])
#     for word in pro_query :
#         work_set = inverted_index.get(word)
#         result_set = result_set & work_set
#     return result_set

# print(basic_search1(inverted_index , 'the tragedy'))
# print(basic_search1(inverted_index , 'the antony'))
# print(basic_search1(inverted_index , 'the ENDS'))


{0, 1, 2, 3, 4, 6, 7}
{0, 3, 4, 7}
{0, 1, 2, 4, 7}


In [ ]:
# Bug 1 : doc.words(file_ids = doc) wrong corpus_name calls this func
# Bug 2 : stopword was a list and searching in list is bad O(N) set made into a set

In [51]:
# Complete function that takes the fileids and creates the inverted index

def create_inv_idx(corpus_name,lang):
    inverted_index = {}
    file_lst = corpus_name.fileids()
    stopword = set(stopwords.words(lang))
    for id , doc in enumerate(file_lst):
        raw_words = corpus_name.words(fileid = doc)
        pro_words = [word.lower() for word in raw_words if word.lower() not in stopword]
        for word in pro_words:
            if word not in inverted_index : inverted_index[word] = {id}
            else : inverted_index[word].add(id)
            
    return inverted_index
        
idx = create_inv_idx(shakespeare,'english')        

In [ ]:


# print(basic_search1(idx , 'the tragedy'))
# print(basic_search1(idx , 'the antony'))
# print(basic_search1(idx , 'the ENDS'))


{0, 1, 2, 3, 4, 6, 7}
{0, 3, 4, 7}
{0, 1, 2, 4, 7}


In [53]:
# query = 'a and b or c and d not e'
# lst_query = query.lower().split()
# oper = [op for op in lst_query if op in {'and','or','not'}]
# print(oper)

In [54]:
# Bug 1 : Not is problemetic operation as it is unary operation so i need to precompute the not oper
# on the words right of not

In [55]:
from collections import deque
all_docs = set(range(len(shakespeare.fileids())))
def basic_search2(idx, query):
    operators = {'and', 'or', 'not'}
    # tokenize query, keep operators even though they're stopwords
    lst_query = query.lower().split()
    pro_query = deque([
        word for word in lst_query
        if word in operators or word in idx
    ])
    if not pro_query:
        return set()
    # ---- Pass 1: resolve NOT immediately, build two clean deques ----
    terms_deque = deque()
    ops_deque = deque()
    while pro_query:
        word = pro_query.popleft()
        if word == 'not':
            if not pro_query:
                break  # malformed query: 'not' with nothing after it
            next_word = pro_query.popleft()
            terms_deque.append(all_docs - idx.get(next_word, set()))
        elif word in {'and', 'or'}:
            ops_deque.append(word)
        else:
            terms_deque.append(idx.get(word, set()))
    if not terms_deque:
        return set()
    # ---- Pass 2: reduce left to right using ops_deque ----
    res_set = terms_deque.popleft()
    while ops_deque:
        op = ops_deque.popleft()
        next_set = terms_deque.popleft()
        if op == 'and':
            res_set = res_set & next_set
        elif op == 'or':
            res_set = res_set | next_set
    return res_set

In [59]:


result = basic_search2(idx, "caesar AND NOT brutus Or tempest And tragedy")
query = "caesar AND NOT brutus Or tempest And tragedy"
for quer in query.lower().split():
    print(quer)
    if quer in idx : print(idx[quer])
    
print('\n')
print(result)

result2 = basic_search2(idx, "caesar OR calpurnia")
print(result2)

caesar
{0, 2, 3, 4, 6}
and
not
brutus
{0, 2, 3, 5}
or
tempest
{1, 2, 3, 4, 6, 7}
and
tragedy
{0, 1, 2, 3, 4, 6, 7}


{1, 2, 3, 4, 6, 7}
{0, 2, 3, 4, 6}
